In [1]:
import pandas as pd
import numpy as np
import sqlite3
from utils import features

In [2]:
conn = sqlite3.connect("data/riot_lol.sqlite")

# "PRAGMA table_info(player_ranks);"

query = """
SELECT
    p.match_id,
    p.team_id,
    p.puuid,
    p.win,
    p.champion_id,
    p.team_position,
    r.tier,
    r.division,
    r.league_points,
    r.wins,
    r.losses
FROM participants p
LEFT JOIN player_ranks r
    ON p.puuid = r.puuid
"""

df = pd.read_sql_query(query, conn)

# print(df.columns)
# print(df.columns[df.columns == 'league_points'])

df.dropna(inplace = True)

rf_df = features.build_features(df)
rf_df = features.encoding_categoricals(rf_df)
print(df.head(5))
print('\n\n\n\n')

# print(df.drop_duplicates("puuid")["tier"].value_counts())
# print(df.drop_duplicates("puuid")["division"].value_counts())

         match_id  team_id                                              puuid  \
0  NA1_5494453720      100  BkoKffZBprI9447fyoRLg9qYANCXTGlcZqqcoLC2uLPwZO...   
1  NA1_5494453720      100  QJ04p4VcgKybTJEEtlUbNInOnEg0Cb1Rv5CPhbNR807ifA...   
2  NA1_5494453720      100  0BUa_yL5ZEg6Y_rCSQd95pB8y7MAEThgSNLZd8jtprMdHO...   
3  NA1_5494453720      100  7orc7p6ptbGwN4j1ibBz9beGiKdqfJ4qCvY6H0v7p9cWUk...   
4  NA1_5494453720      100  mPlS9w6hsowbBuJrzTmmKuSeEyAkl2DXetSvXMn5aVpn4J...   

   win  champion_id team_position         tier division  league_points   wins  \
0    1          126           TOP   CHALLENGER        I         2477.0  293.0   
1    1          238        JUNGLE  GRANDMASTER        I          834.0   72.0   
2    1          112        MIDDLE  GRANDMASTER        I          886.0  135.0   
3    1           81        BOTTOM   CHALLENGER        I         2205.0  125.0   
4    1           34       UTILITY  GRANDMASTER        I          769.0  125.0   

   losses   winrate  
0   

In [7]:
X_train, X_test, y_train, y_test = features.train_test_val_sets(rf_df)

features.random_forest_train(X_train, X_test, y_train, y_test, estimators = 2000)

Random Forest Accuracy at 2000 estimators: 0.55


In [ ]:
# Using GB
gb_df = features.build_features(df, gb = True)
X_train, X_test, y_train, y_test = features.train_test_val_sets(gb_df, gb = True)

iterations = [1000, 2000, 5000]
learning_rates = [0.01, 0.05, 0.10]

for iter in iterations:
    for learn_rate in learning_rates:
        features.cat_boost_train(X_train, X_test, y_train, y_test, iters = iter, lr = learn_rate)

Cat Boost Accuracy at 1000 iterations, lr = 0.01 : 0.55
Cat Boost Accuracy at 1000 iterations, lr = 0.05 : 0.57
Cat Boost Accuracy at 1000 iterations, lr = 0.1 : 0.59
Cat Boost Accuracy at 2000 iterations, lr = 0.01 : 0.56
Cat Boost Accuracy at 2000 iterations, lr = 0.05 : 0.59
Cat Boost Accuracy at 2000 iterations, lr = 0.1 : 0.60
Cat Boost Accuracy at 5000 iterations, lr = 0.01 : 0.57
Cat Boost Accuracy at 5000 iterations, lr = 0.05 : 0.62


In [4]:
# Using GB
gb_df = features.build_features(df, gb = True)
X_train, X_test, y_train, y_test = features.train_test_val_sets(gb_df, gb = True)

features.cat_boost_train(X_train, X_test, y_train, y_test, iters = 5000, lr = 0.1)

Cat Boost Accuracy at 5000 iterations: 0.64
